# Assignment 3 — complete refinement and try alternative Flash

Choose CPU and Runtime > Run all. Step 4 now runs before asking for your key and again after step 3, even when the API is blocked. Its report identifies incomplete input coverage and which candidate method was used.

Step 3 tries Gemini 3.6 Flash as a separate higher-standard-price model using your existing Free Tier project with billing disabled. Original Gemini 3.8 outputs are archived and never mixed into the new model score. This is a new 50-invoice model run, not a continuation claiming mixed outputs are one model. Quota errors stop API calls; the notebook cannot remove Google quotas. All saved progress, refined CSV, evaluation and honest completion status download at the end.

Enter your key only in the private prompt. Source invoice text is sent to Google; ground truth is only used for evaluation. Download the executed notebook via File > Download > Download .ipynb. [Published pricing](https://ai.google.dev/gemini-api/docs/pricing).

Diagnostic update: restores 19 saved Flash 3.6 results. On failure, the output and ZIP show Google’s actual quota metric, limit value and retry delay when supplied. It no longer labels every 429 as a daily quota. Short waits are retried only when the provider identifies them. No billing is enabled.

Batch recovery update: reuses the 19 verified results and sends up to five pending invoices per request (seven successful requests for the remaining 31). Runs one pass, with at most two attempts per batch and a stop after three consecutive failed batches. Service availability and quota are checked by the actual run; completion is not guaranteed.


In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','pypdf>=5,<7','requests'],check=True)
print('Ready. No GPU or large model download is needed.')


In [ ]:
from pathlib import Path
import io,json,zipfile,requests,os,hashlib,datetime,importlib.metadata,shutil,csv,getpass
WORK=Path('/content/assignment3_recovery_run');WORK.mkdir(exist_ok=True)
r=requests.get('https://codeload.github.com/MVEST22/invoice-extraction-assignment/zip/refs/heads/main',timeout=120)
r.raise_for_status()
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    for entry in z.infolist():
        p=Path(entry.filename)
        if p.is_absolute() or '..' in p.parts:raise ValueError('Unexpected source archive path')
    z.extractall(WORK)
ROOT=WORK/'invoice-extraction-assignment-main';os.chdir(ROOT)
subprocess.run([sys.executable,'scripts/setup_data.py'],check=True)
subprocess.run([sys.executable,'scripts/preflight.py'],check=True)
sys.path.insert(0,str(ROOT/'methods'))
print('50 source invoices ready; answers are reserved for evaluation.')

# Restore evidence, never overwrite newer local progress or source code.
with zipfile.ZipFile(ROOT/'runs/2026-11-02_partial/checkpoint.zip') as z:
    for entry in z.infolist():
        if entry.is_dir():continue
        parts=Path(entry.filename).parts
        if not parts or parts[0]!='outputs' or '..' in parts:continue
        target=ROOT/entry.filename
        # Repair empty files created for ZIP directory entries by the previous template.
        for parent in reversed(target.parents):
            if parent==ROOT or ROOT in parent.parents:
                if parent.is_file():
                    backup=parent.with_name(parent.name+'.old_zip_directory_entry')
                    if backup.exists():raise RuntimeError('Unexpected backup collision: '+str(backup))
                    parent.rename(backup)
                parent.mkdir(exist_ok=True)
        if not target.exists():
            target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(z.read(entry))
manifest=json.loads((ROOT/'runs/2026-11-02_partial/run_manifest.json').read_text())
for name,digest in manifest['input_sha256'].items():
    assert hashlib.sha256((ROOT/'invoices'/name).read_bytes()).hexdigest()==digest
print('Saved results restored. Method 2 will make no new API requests.')

# Restore the sixth genuine response from the latest returned checkpoint.
seed=ROOT/'runs/2026-11-02_partial/extra_raw'
for source in seed.glob('*.json'):
    target=ROOT/'outputs/raw/method3_expensive'/source.name
    if not target.exists():shutil.copyfile(source,target)

# Restore the 19 successful Flash 3.6 checkpoints returned on 2 November.
# Existing files are preserved, including any newer progress in this runtime.
resume_seed=ROOT/'runs/2026-11-02_flash36_resume'
resume_target=ROOT/'outputs/raw/method3_expensive'
resume_target.mkdir(parents=True,exist_ok=True)
for source in resume_seed.glob('gemini_*.json'):
    target=resume_target/source.name
    if not target.exists():shutil.copyfile(source,target)
print('Flash 3.6 recovery checkpoints installed; matching successful responses will be reused.')


In [ ]:
subprocess.run([sys.executable,'methods/method1_programmatic.py'],check=True)
subprocess.run([sys.executable,'evaluation/compare_results.py'],check=True)


In [ ]:
# Step 4 can run using available candidates. Its provenance log identifies the base method.
for script in ['methods/method4_refined.py','evaluation/compare_results.py','scripts/calculate_costs.py','scripts/build_db.py','scripts/build_report.py']:
    subprocess.run([sys.executable,script],check=True)
print((ROOT/'REPORT.md').read_text())


In [ ]:
# Explicit reload avoids stale code if you rerun cells in an existing session.
import importlib,gemini_extract
importlib.reload(gemini_extract)
MODEL_PAIR={'method2_low_cost':'gemini-3.5-flash-lite','method3_expensive':'gemini-3.8-flash'}
KEY=getpass.getpass('Gemini Free Tier key (private): ').strip()
if not KEY:raise RuntimeError('Enter your key in the private prompt.')


In [ ]:
RUN_ERROR=None
try:
    # The original 3.8 run is quota-blocked. Attempt a separate 3.6 run, without mixing models.
    ALT='gemini-3.6-flash'
    access=requests.get(gemini_extract.API+'/models/'+ALT,headers={'x-goog-api-key':KEY},timeout=60)
    if not access.ok:
        diagnostic=gemini_extract.provider_error(access,KEY)
        (ROOT/'outputs/model_access_error.json').write_text(json.dumps(diagnostic,indent=2))
        print('Google error details (key removed):',json.dumps(diagnostic,ensure_ascii=False))
        raise RuntimeError('Model access HTTP '+str(access.status_code)+': '+diagnostic.get('message','No detailed provider message.'))
    if 'generateContent' not in access.json().get('supportedGenerationMethods',[]):
        raise RuntimeError('Alternative model does not advertise text generation.')
    archived=ROOT/'outputs/archived_gemini_3_8';archived.mkdir(exist_ok=True)
    rawdir=ROOT/'outputs/raw/method3_expensive';rawdir.mkdir(parents=True,exist_ok=True)
    for path in rawdir.glob('gemini_*.json'):
        saved=json.loads(path.read_text())
        if saved.get('model')=='gemini-3.8-flash':
            shutil.copyfile(path,archived/path.name)
            path.unlink()
    up=ROOT/'outputs/method3_expensive_usage.json'
    oldusage=json.loads(up.read_text()) if up.exists() else []
    if oldusage and oldusage[0].get('model')=='gemini-3.8-flash':
        for name in ['method3_expensive.csv','method3_expensive_usage.json']:
            shutil.copyfile(ROOT/'outputs'/name,archived/name)
    MODEL_PAIR['method3_expensive']=ALT
    print('Step 3: separate Gemini 3.6 Flash run. Original six 3.8 results are archived.')
    print('Resuming saved results; up to five pending invoices per request. One pass only.')
    gemini_extract.run(KEY,'method3_expensive',ALT,batch_size=5,continue_transient=True)
except (RuntimeError,requests.RequestException) as exc:
    # Do not print provider exception strings: only the safe message/status.
    RUN_ERROR=str(exc) if isinstance(exc,RuntimeError) else 'Network connection failed.'
    print('Step 3 remains incomplete:',RUN_ERROR)
finally:
    KEY=None
    print('Running step 4 and packaging results even if step 3 was blocked.')


In [ ]:
# Step 4 can run using available candidates. Its provenance log identifies the base method.
for script in ['methods/method4_refined.py','evaluation/compare_results.py','scripts/calculate_costs.py','scripts/build_db.py','scripts/build_report.py']:
    subprocess.run([sys.executable,script],check=True)
print((ROOT/'REPORT.md').read_text())


In [ ]:
from google.colab import files
DEST=WORK/'results_package';DEST.mkdir(exist_ok=True)
for directory in ['outputs','evaluation','methods','scripts']:
    shutil.copytree(ROOT/directory,DEST/directory,dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__'))
for name in ['REPORT.md','README.md','requirements.txt','run_pipeline.py','tests_local.py','invoice_results.db','Assignment_3_Recovery_Colab.ipynb']:
    if (ROOT/name).exists():shutil.copyfile(ROOT/name,DEST/name)
status={}
for method in ['method1_programmatic','method2_low_cost','method3_expensive','method4_refined']:
    path=ROOT/'outputs'/(method+'.csv')
    if path.exists():
        with path.open(encoding='utf-8-sig',newline='') as f:status[method]=len(list(csv.DictReader(f)))
    else:status[method]=0
costpath=ROOT/'evaluation/costs.json'
costs=json.loads(costpath.read_text()) if costpath.exists() else []
manifest={'packaged_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'rows':status,
    'complete':all(n==50 for n in status.values()),'refinement_provisional':status.get('method3_expensive',0)<50,'step3_error':globals().get('RUN_ERROR'),'method':'Gemini API on CPU Colab; no local model inference',
    'models':{r['method']:r['model'] for r in costs},'billing_basis':'Eligible Free Tier project chosen by user; billing disabled; not independently verified by inference API.',
    'free_tier_charge_usd':0,'usage':costs,
    'packages':{p:importlib.metadata.version(p) for p in ['pypdf','requests']},
    'input_sha256':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((ROOT/'invoices').glob('*.pdf'))},
    'code_sha256':{str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for d in ['methods','scripts','evaluation'] for p in (ROOT/d).glob('*.py')}}
(DEST/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
(DEST/'EXECUTION_NOTE.txt').write_text('The included notebook is an unexecuted source template. Download your actual executed notebook separately using File > Download > Download .ipynb. Return that notebook and this ZIP. Check manifest complete and row counts. API key is not included.')
archive=shutil.make_archive(str(WORK/'assignment3_results'),'zip',DEST)
print('Rows per method:',status,'Complete:',manifest['complete'])
files.download(archive)
